# Aplicación de CNN propio

## Configuración de entorno

In [4]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import os

# Configuración de rutas
BASE_DIR = Path("data")
IMG_DIR = BASE_DIR / "processed" / "train"
CSV_PATH = BASE_DIR / "processed" / "val" / "data_300.csv"

# Parámetros de visualización
PLOTS_DIR = Path("reports/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {
    1: "auto", 2: "combi", 3: "microbus", 4: "minibus", 
    5: "omnibus", 6: "articulado", 7: "camion", 8: "mototaxi", 9: "motocicleta"
}

## Etapa 1: Preparación del Dataset de Clasificación (Cropping)

En lugar de pasar la imagen completa de la intersección, crearemos un nuevo dataset donde cada imagen sea un vehículo individual.


In [5]:
import cv2
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

def create_classification_dataset(df, input_img_dir, output_dir):
    """Extrae cada vehículo y lo guarda en una carpeta según su clase."""
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = input_img_dir / f"{row['Id']}.jpg"
        if not img_path.exists() or row['count'] == 0: continue
        
        img = cv2.imread(str(img_path))
        for i, obj in enumerate(row['objects']):
            # Extraer recorte (Crop) simple basado en cx, cy, w, h
            # (Ignoramos el ángulo para esta CNN base para simplificar)
            x1 = int(obj['cx'] - obj['w']/2)
            y1 = int(obj['cy'] - obj['h']/2)
            x2 = int(obj['cx'] + obj['w']/2)
            y2 = int(obj['cy'] + obj['h']/2)
            
            crop = img[max(0,y1):y2, max(0,x1):x2]
            if crop.size == 0: continue
            
            # Guardar en carpeta correspondiente: output/class_id/image_id_n.jpg
            class_path = output_dir / str(obj['class_id'])
            class_path.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(class_path / f"{row['Id']}_{i}.jpg"), crop)

# Ejecución sobre la muestra de 300 imágenes
CROP_DIR = Path("data/classification_baseline")
create_classification_dataset(df, IMG_DIR, CROP_DIR)


NameError: name 'df' is not defined